In [1]:
import json
import random
from pathlib import Path
from typing import Any

from datasets import Dataset, DatasetDict, load_dataset
from transformers import AutoTokenizer

hf_token =json.load(open(Path("./config.json")))["hg_access_token"]
cache_dir =json.load(open(Path("./config.json")))["cache_dir"]
dataset_path=json.load(open(Path("./config.json")))["dataset_path"]

/home/erfan/miniconda3/envs/hf/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:

# ============================================================
# Configuration
# ============================================================

OUTPUT_PATH = Path("./dataset/huggingface/qwen3_tool_calling")
MODEL_NAME = "Qwen/Qwen3-0.6B"

TEST_SIZE = 0.10
SEED = 42

# Set this to the key identifying paraphrases of the same example.
# Example:
# metadata = {"canonical_id": "fundamentals_000123"}
GROUP_KEY = "canonical_id"


SYSTEM_PROMPT = """
You are a financial function-calling model.

Convert the user's request into exactly one valid JSON object.

Supported functions:
- get_fundamentals
- get_weekly_prices
- screen_fundamentals

Rules:
- Select the correct function.
- Extract and normalize every required argument.
- Preserve the association between companies and their requested metrics.
- Do not answer the financial question yourself.
- Do not calculate or invent financial values.
- Do not explain your reasoning.
- Do not generate SQL.
- Output only valid JSON.
""".strip()



In [4]:

# ============================================================
# Load tokenizer
# ============================================================



tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, token=hf_token, cache_dir=cache_dir)


# ============================================================
# Load JSONL
# ============================================================


raw_dataset = load_dataset(
    "json",
    data_files=str(dataset_path),
    split="train",
)

required_columns = {
    "query",
    "output_str",
    "output_json",
    "metadata",
}

missing_columns = required_columns - set(raw_dataset.column_names)

if missing_columns:
    raise ValueError(
        f"Missing required columns: {sorted(missing_columns)}"
    )



In [6]:

# ============================================================
# JSON validation and normalization
# ============================================================

def parse_json_value(value: Any) -> dict:
    """
    Convert either a JSON string or Python dictionary into a dictionary.
    """
    if isinstance(value, dict):
        return value

    if isinstance(value, str):
        parsed = json.loads(value)

        if not isinstance(parsed, dict):
            raise ValueError("Tool-call output must be a JSON object.")

        return parsed

    raise TypeError(
        f"Expected output to be dict or JSON string, got {type(value).__name__}"
    )


def validate_example(example: dict, index: int) -> dict:
    query = example["query"]

    if not isinstance(query, str) or not query.strip():
        raise ValueError(f"Example {index}: query is empty or invalid.")

    try:
        output_from_str = parse_json_value(example["output_str"])
    except Exception as exc:
        raise ValueError(
            f"Example {index}: invalid output_str: {exc}"
        ) from exc

    try:
        output_from_json = parse_json_value(example["output_json"])
    except Exception as exc:
        raise ValueError(
            f"Example {index}: invalid output_json: {exc}"
        ) from exc

    # Ensure that output_str and output_json represent the same object.
    if output_from_str != output_from_json:
        raise ValueError(
            f"Example {index}: output_str and output_json do not match.\n"
            f"output_str: {output_from_str}\n"
            f"output_json: {output_from_json}"
        )

    # Compact deterministic JSON target.
    normalized_output = json.dumps(
        output_from_json,
        ensure_ascii=False,
        separators=(",", ":"),
        sort_keys=False,
    )

    metadata = example["metadata"]

    if metadata is None:
        metadata = {}

    if not isinstance(metadata, dict):
        raise TypeError(
            f"Example {index}: metadata must be a dictionary."
        )

    return {
        "query": query.strip(),
        "completion": normalized_output,
        "metadata": metadata,
        "source_index": index,
    }


validated_dataset = raw_dataset.map(
    validate_example,
    with_indices=True,
    remove_columns=raw_dataset.column_names,
    desc="Validating examples",
)


# ============================================================
# Split without leaking paraphrases
# ============================================================

def group_train_test_split(
    dataset: Dataset,
    group_key: str,
    test_size: float,
    seed: int,
) -> DatasetDict:
    """
    Split by metadata[group_key].

    All examples with the same canonical ID remain in the same split.
    If the group key is unavailable, each row becomes its own group.
    """
    groups: dict[str, list[int]] = {}

    for row_index, example in enumerate(dataset):
        metadata = example.get("metadata") or {}
        group_id = metadata.get(group_key)

        if group_id is None:
            # This row becomes an independent group.
            group_id = f"row_{example['source_index']}"

        group_id = str(group_id)
        groups.setdefault(group_id, []).append(row_index)

    group_ids = list(groups.keys())

    if len(group_ids) < 2:
        raise ValueError(
            "At least two distinct groups are required for train/test splitting."
        )

    rng = random.Random(seed)
    rng.shuffle(group_ids)

    number_of_test_groups = max(
        1,
        round(len(group_ids) * test_size),
    )

    # Ensure that at least one group remains in training.
    number_of_test_groups = min(
        number_of_test_groups,
        len(group_ids) - 1,
    )

    test_group_ids = set(group_ids[:number_of_test_groups])

    train_indices: list[int] = []
    test_indices: list[int] = []

    for group_id, indices in groups.items():
        if group_id in test_group_ids:
            test_indices.extend(indices)
        else:
            train_indices.extend(indices)

    return DatasetDict(
        {
            "train": dataset.select(train_indices),
            "test": dataset.select(test_indices),
        }
    )


dataset_splits = group_train_test_split(
    dataset=validated_dataset,
    group_key=GROUP_KEY,
    test_size=TEST_SIZE,
    seed=SEED,
)


# ============================================================
# Build Qwen conversation format
# ============================================================

def build_training_example(example: dict) -> dict:
    messages = [
        {
            "role": "system",
            "content": SYSTEM_PROMPT,
        },
        {
            "role": "user",
            "content": example["query"],
        },
        {
            "role": "assistant",
            "content": example["completion"],
        },
    ]

    # Fully formatted text for trainers expecting a `text` column.
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False,
        enable_thinking=False,
    )

    return {
        "messages": messages,
        "text": text,
    }


dataset_splits = dataset_splits.map(
    build_training_example,
    desc="Applying Qwen chat template",
)


# Keep useful columns.
dataset_splits = DatasetDict(
    {
        split_name: split_dataset.select_columns(
            [
                "messages",
                "text",
                "query",
                "completion",
                "metadata",
                "source_index",
            ]
        )
        for split_name, split_dataset in dataset_splits.items()
    }
)


# ============================================================
# Save dataset
# ============================================================

OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

dataset_splits.save_to_disk(str(OUTPUT_PATH))


# ============================================================
# Inspection
# ============================================================

print(dataset_splits)

print("\nTrain size:", len(dataset_splits["train"]))
print("Test size:", len(dataset_splits["test"]))

print("\nMessages representation:")
print(
    json.dumps(
        dataset_splits["train"][0]["messages"],
        indent=2,
        ensure_ascii=False,
    )
)

print("\nFormatted training text:")
print(dataset_splits["train"][0]["text"])

Saving the dataset (1/1 shards): 100%|██████████| 310/310 [00:00<00:00, 100442.97 examples/s]

DatasetDict({
    train: Dataset({
        features: ['messages', 'text', 'query', 'completion', 'metadata', 'source_index'],
        num_rows: 2788
    })
    test: Dataset({
        features: ['messages', 'text', 'query', 'completion', 'metadata', 'source_index'],
        num_rows: 310
    })
})

Train size: 2788
Test size: 310

Messages representation:
[
  {
    "role": "system",
    "content": "You are a financial function-calling model.\n\nConvert the user's request into exactly one valid JSON object.\n\nSupported functions:\n- get_fundamentals\n- get_weekly_prices\n- screen_fundamentals\n\nRules:\n- Select the correct function.\n- Extract and normalize every required argument.\n- Preserve the association between companies and their requested metrics.\n- Do not answer the financial question yourself.\n- Do not calculate or invent financial values.\n- Do not explain your reasoning.\n- Do not generate SQL.\n- Output only valid JSON."
  },
  {
    "role": "user",
    "content": "What